In [1]:
# 檔案: mask_to_pseudolabel.py
# 描述: 將二元的黑白遮罩 (binary masks) 轉換為多標籤的偽標籤格式。
#      在這個格式中，每個獨立的連通前景區域都會被賦予一個唯一的整數ID。

import os
import glob
import numpy as np
from skimage import io, measure
from tqdm import tqdm # 導入 tqdm 來顯示進度條

def convert_mask_to_pseudolabel(input_mask_path, output_label_path):
    """
    讀取一張二元遮罩，找到其中的所有連通區域，並將它們轉換為帶有唯一ID的偽標籤圖。

    Args:
        input_mask_path (str): 輸入的二元遮罩檔案路徑。
        output_label_path (str): 輸出的偽標籤檔案路徑。
    """
    try:
        # 1. 讀取遮罩圖像
        # as_gray=True 確保即使遮罩被存為RGB格式，也能被正確讀取為單通道
        mask = io.imread(input_mask_path, as_gray=True)

        # 2. 二值化 (Binarization)
        #    將圖像轉換為嚴格的 0 和 1。
        #    假設前景像素值大於一個閾值 (例如 128)。
        binary_mask = (mask > 128).astype(np.uint8)

        # 3. 尋找連通區域 (Connected Component Labeling)
        #    這是最關鍵的一步。`measure.label` 會掃描二值化圖像，
        #    找到所有獨立的白色區域，並為每個區域分配一個從1開始的唯一整數。
        #    `background=0` 指定了值為0的像素是背景。
        #    `connectivity=2` (或8-connectivity) 確保對角線相鄰的像素也被視為連接。
        pseudolabel_map, num_labels = measure.label(binary_mask, background=0, connectivity=2, return_num=True)

        # 4. 儲存結果
        if num_labels > 0:
            # 儲存為 16-bit PNG 格式，以支援超過 255 個標籤。
            # 這是為了以防萬一某張圖中有很多微小的獨立區域。
            io.imsave(output_label_path, pseudolabel_map.astype(np.uint16), check_contrast=False)
        else:
            # 如果圖像中沒有任何前景，就創建一個全黑的圖像。
            io.imsave(output_label_path, np.zeros_like(mask, dtype=np.uint16), check_contrast=False)
        
        return num_labels

    except Exception as e:
        print(f"處理檔案 {os.path.basename(input_mask_path)} 時發生錯誤: {e}")
        return 0


def main():
    """
    主函式，用於設定路徑並遍歷所有遮罩檔案進行轉換。
    """
    # --- 在此處設定您的路徑 ---
    
    # 輸入: 包含原始二元黑白遮罩的資料夾
    INPUT_MASK_DIR = '/home/U116med/data/polyp/TrainDataset/masks/'
    
    # 輸出: 儲存轉換後的多標籤偽標籤的資料夾
    OUTPUT_PSEUDOLABEL_DIR = './data/polypdata/pseudolabel_supervised/'

    # --- 流程開始 ---

    print(f"輸入遮罩目錄: {INPUT_MASK_DIR}")
    print(f"輸出偽標籤目錄: {OUTPUT_PSEUDOLABEL_DIR}")

    # 建立輸出資料夾 (如果不存在)
    os.makedirs(OUTPUT_PSEUDOLABEL_DIR, exist_ok=True)

    # 找到所有需要轉換的遮罩檔案 (支援 .png 和 .jpg)
    mask_paths = sorted(glob.glob(os.path.join(INPUT_MASK_DIR, '*.png')))
    mask_paths.extend(sorted(glob.glob(os.path.join(INPUT_MASK_DIR, '*.jpg'))))
    
    if not mask_paths:
        print(f"錯誤: 在 '{INPUT_MASK_DIR}' 中找不到任何遮罩檔案。請檢查路徑。")
        return

    print(f"找到 {len(mask_paths)} 個遮罩檔案，開始轉換...")
    
    # 使用 tqdm 來顯示進度條
    for mask_path in tqdm(mask_paths, desc="轉換進度"):
        # 構建輸出的檔案名稱
        base_name = os.path.basename(mask_path)
        # 確保輸出檔名與我們的 SSL Dataset 期望的格式一致 (_label.png)
        output_name = os.path.splitext(base_name)[0] + '_label.png'
        output_path = os.path.join(OUTPUT_PSEUDOLABEL_DIR, output_name)
        
        # 執行轉換
        convert_mask_to_pseudolabel(mask_path, output_path)
        
    print("\n所有遮罩檔案已成功轉換為偽標籤格式！")


if __name__ == '__main__':
    # 安裝 tqdm (如果尚未安裝): pip install tqdm
    main()

輸入遮罩目錄: /home/U116med/data/polyp/TrainDataset/masks/
輸出偽標籤目錄: ./data/polypdata/pseudolabel_supervised/
找到 1450 個遮罩檔案，開始轉換...


轉換進度: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1450/1450 [00:11<00:00, 126.69it/s]


所有遮罩檔案已成功轉換為偽標籤格式！
